# Assignment 1: Chronicles Across the Ages

**Starter notebook.** This gives you the plumbing (loading data, calling your local MiniLM
server) already wired up, so you can spend your time on the actual continual-learning and
retrieval logic rather than JSON parsing and HTTP calls. Every `# TODO` marker is something
you need to implement yourself, see the assignment brief for the exact task requirements and
mark weights. Where a TODO describes what's needed rather than giving you an empty function to
fill in, that's deliberate, you're free to structure that part as a function, a class, or
straight-line code, whatever fits how you think about it.

## Before you start

1. Follow `assignment-01-lmstudio-setup-tutorial.md` to get `all-MiniLM-L6-v2` running, either locally via LM Studio, or remotely via Hugging Face's `transformers`/`sentence-transformers` (e.g. in Colab), your choice. The `get_embeddings()` helper below assumes the local LM Studio path by default; if you go the remote route instead, adapt it (or compute your embeddings in Colab and bring the resulting vectors back into this notebook, e.g. as a saved `.npy` file)
2. For Part 2(c), also load a small language model (see the setup guide for suggestions) or use the Hugging Face alternative described near the end of that same guide if you'd rather not install a second model locally. Either is a fully supported option
3. Put this notebook in the same folder as `passages.json`, `questions.json`, and `baseline_questions.json` (or edit `DATA_DIR` below), then run cells top to bottom, filling in each TODO as you go. Good luck!!

**As provided, this notebook calls `http://localhost:1234` for both MiniLM and the generation model**, so it won't run as-is in a cloud/hosted environment. If you're running either model remotely instead, adjust `get_embeddings()` and/or the chat call accordingly.

## Setup 

You shouldn't need to change this section, aside from the model IDs
Also, if you are using Hugging face or other way to access the models, then you will need to adapt this

In [12]:
import json
import os
import re

import numpy as np
import pandas as pd
from openai import OpenAI
# from sklearn.neural_network import MLPClassifier
# from IPython.display import display

# --- Edit these for your machine ---
DATA_DIR = "data/"  # folder containing passages.json / questions.json / baseline_questions.json
LMSTUDIO_URL = "http://localhost:1234/v1"
EMBED_MODEL_ID = "text-embedding-all-minilm-l6-v2-embedding@q5_k_s"  # exact ID LM Studio shows for MiniLM
CHAT_MODEL_ID = "llama-3.2-3b-instruct"  # needed for Part 2(c) (leave as-is if using the Colab alternative instead)
# ------------------------------------

API_KEY = os.environ.get("LMSTUDIO_API_KEY", "lm-studio")
client = OpenAI(base_url=LMSTUDIO_URL, api_key=API_KEY)


def get_embeddings(text_list, batch_size=16):
    """Batch-fetch MiniLM embeddings from the local LM Studio server. Returns an
    (len(text_list) x 384) numpy array, one row per input string, in the same order."""
    out = []
    for i in range(0, len(text_list), batch_size):
        batch = [t.replace("\n", " ") for t in text_list[i:i + batch_size]]
        resp = client.embeddings.create(input=batch, model=EMBED_MODEL_ID)
        out.extend([d.embedding for d in resp.data])
    return np.array(out)


pd.set_option("display.width", 120)
np.set_printoptions(suppress=True)


In [14]:
with open(f"{DATA_DIR}/passages.json") as f:
    passages = json.load(f)
with open(f"{DATA_DIR}/questions.json") as f:
    questions = json.load(f)
with open(f"{DATA_DIR}/baseline_questions.json") as f:
    baseline_questions = json.load(f)

ERAS = ["The Ember Age", "The Steel Age", "The Hallow Age", "The Tide Age", "The Cinder Age", "The Lumen Age"]
EXPERIENCES = [
    ["The Ember Age", "The Steel Age"],
    ["The Hallow Age", "The Tide Age"],
    ["The Cinder Age", "The Lumen Age"],
]
TIERS = ["easy", "medium", "hard"]
ERA_TO_LABEL = {era: i for i, era in enumerate(ERAS)}


def by_era(items, era):
    return [x for x in items if x["era"] == era]


def texts(items):
    return [x["text"] if "text" in x else x["question"] for x in items]


print(f"{len(passages)} passages, {len(questions)} questions, "
      f"{len(baseline_questions)} baseline (real-history) questions, {len(ERAS)} eras")



180 passages, 234 questions, 96 baseline (real-history) questions, 6 eras


# Part 1: Continual Learning: The Era Router (55%)

See the assignment brief for the full task description. Quick reminder of the setup: a small
classifier (e.g. an `MLPClassifier` over MiniLM embeddings) is trained **sequentially** across the
3 Experiences in `EXPERIENCES` above (2 new eras per Experience), and evaluated after each
Experience on every era introduced so far.


### Data prep: stratified per-era train/test split, then embeddings

In [17]:
TEST_FRAC = 0.3  # held out per tier, per era

# TODO: for each era, split its passages (given as `items`) into a training pool and a
# held-out test pool, STRATIFIED BY DIFFICULTY TIER, i.e. each tier (easy/medium/hard) should be
# split ~70/30 independently, not the era as a whole, so a small test set can't end up with zero
# examples of some tier. Use TEST_FRAC above and np.random.RandomState(seed) for reproducibility.
#
# One way to organise this: a stratified_split(items, seed) -> (train_items, test_items) function
# you call once per era. You don't have to structure it that way, just make sure you end up with
# a train/test split per era that you can feed into the embedding step below.

def stratified_split(items, seed):
    rng = np.random.RandomState(seed)
    train_items, test_items = [], []

    for tier in TIERS:
        tier_items = [it for it in items if it["difficulty"] == tier]
        order = rng.permutation(len(tier_items))
        tier_items = [tier_items[i] for i in order]

        n_test = max(1, int(round(len(tier_items) * TEST_FRAC)))
        test_items.extend(tier_items[:n_test])
        train_items.extend(tier_items[n_test:])

    rng.shuffle(train_items)
    rng.shuffle(test_items)
    return train_items, test_items

# quick sanity check
example_train, example_test = stratified_split(by_era(passages, ERAS[0]), seed=335)
print("Example era:", ERAS[0])
print("Train:", len(example_train), " Test:", len(example_test))
for tier in TIERS:
    tr = sum(it["difficulty"] == tier for it in example_train)
    te = sum(it["difficulty"] == tier for it in example_test)
    print(f"  {tier:>6}: train={tr}, test={te}")

In [ ]:
# TODO: for every era, split its passages the way described above, then fetch MiniLM
# embeddings (via get_embeddings) for the resulting train passages, test passages, and that
# era's questions. Store them somewhere you can look up by era for the training loop below, e.g.
# three dicts keyed by era name -> array (era_train_emb, era_test_emb, era_question_emb), and
# similarly keep the *items* (not just embeddings) for era_train_items / era_test_items since
# you'll need each item's "difficulty" field later for the tier breakdown.
#
# Print each era's train/test/question counts as you go, and the embedding dimension at the end,
# as a sanity check (should be 384).

### Sequential training loop

You'll reuse this across parts (a) and (b): a naive condition (no replay) for part (a), and full
replay / bounded replay conditions for part (b). Consider writing one function that takes a
`condition` argument rather than three separate copies, but structure it however makes sense to
you.


Requirements (see assignment brief for full detail):
* At each Experience, train on that Experience's 2 new eras' training passages.
* condition == "naive": never revisit earlier Experiences' raw passages at all.
* condition == "full_replay": at every Experience, mix in EVERY previously-seen training passage alongside the new Experience's data.
* condition == "bounded_replay": at every Experience, mix in only a fixed-size buffer (`buffer_size` passages per earlier era, sampled once when that era was learned) not a percentage of the dataset.
* Use MLPClassifier.partial_fit (if sklearn is used) with `classes=ALL_CLASSES` passed on every call (not just the first) so the output layer has room for eras that haven't appeared yet, this is what makes it genuinely class-incremental.
* After EACH Experience, evaluate the current model on EVERY era introduced so far (not just the newest), using both held-out questions and held-out passages, and also record accuracy broken down by difficulty tier (using the held-out passages' "difficulty" field).

Notice that we have the `condition` in here, but maybe you want separate functions for each (there will be some duplication in that case), which is fine

In [ ]:
N_ITERS = 1500  # a reasonable number of partial_fit calls per training stage, tune if you like, but that is not a requirement
ALL_CLASSES = list(range(len(ERAS)))

# TODO: train ONE classifier sequentially across the 3 Experiences in EXPERIENCES, under
# whichever condition you're running (see the requirements above). By the end you should have,
# for every combination of "Experience just finished" and "era", the held-out QUESTION accuracy,
# the held-out PASSAGE accuracy, and the accuracy broken down by difficulty tier (easy/medium/
# hard), for every era introduced so far at that point (NaN for eras not yet introduced).
#
# One way to organise this: a run_experience_sequence(condition, seed, buffer_size=None) function
# returning (question_acc, passage_acc, tier_acc) as (3 Experiences x 6 eras) arrays and a
# {"easy": array, "medium": array, "hard": array} dict, matching what matrix_df() and
# peak_accuracy() below expect. You don't have to structure it this way, just make sure whatever
# you build feeds into the reporting below in that shape.

Averaging over a few seeds is optional but recommended: run your training loop above over
`n_seeds` different seeds and average the results (np.nanmean), since the assignment's Notes
section flags that single-seed results can be noisy at this dataset size. Same
(question_acc, passage_acc, tier_acc) shape as above, averaged across seeds.

In [ ]:
# TODO, optional but recommended: average your training loop's results over n_seeds
# different seeds (np.nanmean), for the same (question_acc, passage_acc, tier_acc) shape.


def matrix_df(mat):
    """Provided: wraps a (3 x 6) accuracy matrix as a labelled DataFrame for display."""
    return pd.DataFrame(mat, index=[f"after Experience {i + 1}" for i in range(mat.shape[0])], columns=ERAS).round(3)


# Each Experience introduces 2 ages (eras), so an era's "peak" (right after it was first learned) is not
# on the matrix diagonal, it's whichever Experience first introduced that era. Provided, since
# it's bookkeeping rather than the interesting part of the task
ERA_INTRO_STAGE = {era: stage for stage, exp_eras in enumerate(EXPERIENCES) for era in exp_eras}


def peak_accuracy(mat):
    """For each era (column), accuracy right after the Experience that first introduced it."""
    return np.array([mat[ERA_INTRO_STAGE[era], i] for i, era in enumerate(ERAS)])

## (a) Observe Catastrophic Forgetting (20%)

In [ ]:
# TODO: run the naive condition, display the held-out QUESTION accuracy matrix and the
# held-out PASSAGE accuracy matrix (use matrix_df + display).

In [10]:
# TODO: using peak_accuracy() and the final row of your naive-condition question matrix,
# print each era's peak vs. final accuracy and the drop between them, the average drop across
# eras, and the final-Experience accuracy broken down by difficulty tier.


**Discussion (write answer here):** See assignment for the question

## (b) Experience Replay (20%)

**TODO**: choose at least 2 buffer sizes for bounded_replay (we  suggest 3-5 passages/era as a starting point). 

Run "naive", "full_replay", and bounded_replay at each of your chosen buffer sizes, and store all the results somewhere you can compare (e.g. a dict keyed by a condition label)

**TODO**: display the held-out QUESTION accuracy matrix for every condition you ran.

**TODO**: build a summary table (one row per condition) with avg_final_accuracy and avg_forgetting for every condition from the cell above, and display it.

In [ ]:
# TODO: for each condition's question accuracy matrix, compute avg_forgetting (the
# average, across eras, of each era's peak accuracy per peak_accuracy() above, minus its
# accuracy after the final Experience) and avg_final_accuracy (the mean of the final row).
# You'll want both per condition to build the summary table above.

**Discussion (write answer here):** See assignment


## (c) CL Comparison and Interpretation (15%)

No new code strictly required here, this is about interpreting the numbers you already have
from part (b). You may add a code cell below if it helps you compute anything (like the % of the
naive-to-full-replay gap that bounded replay recovers, and the memory cost of bounded vs. full
replay in terms of passages stored)

**Discussion (write answer here):** See assignment


# Part 2: Retrieval-Augmented Generation (45%)

Reuses `passages.json` / `questions.json`


### Setup: full-corpus embeddings (provided)

as expected, this is going to fail unless you have LM Studio with MiniLM running and configured

In [13]:
print("Fetching MiniLM embeddings for the full passage collection and all questions (Part 2)...")
passage_texts = [p["text"] for p in passages]
passage_emb = get_embeddings(passage_texts)

question_texts = [q["question"] for q in questions]
question_emb = get_embeddings(question_texts)

passage_by_id = {p["passage_id"]: p for p in passages}
question_by_id = {q["question_id"]: q for q in questions}
era_passage_idx = {era: [i for i, p in enumerate(passages) if p["era"] == era] for era in ERAS}

print(f"passage_emb shape: {passage_emb.shape}, question_emb shape: {question_emb.shape}")


## (a) Build the Retriever and Evaluate Retrieval Quality (25%)

In [ ]:
K_VALUES = [1, 3, 5]

# TODO: write the ranking step yourself (no LangChain/LlamaIndex, per the assignment's
# Library usage note), ranking candidate passages/chunks by COSINE similarity (not raw dot
# product, normalize both the query and candidate vectors first) to a query embedding, and
# returning the top-k. This is the core piece you'll reuse for both era-restricted and
# full-corpus retrieval below, and again for chunked retrieval in part (b), so it's worth writing
# it once in a way you can call repeatedly, e.g. a cosine_topk(query_vec, matrix, k) ->
# (indices, similarities) function, but however you structure it is fine.

For part (a) you need retrieval evaluation logic that: for every question (optionally
restricted to one difficulty tier), ranks candidate passages by similarity to the question's
embedding (using whatever ranking approach you wrote above), and checks whether the question's
`evidence_passage_id` appears in the top-1 / top-3 / top-5. It also needs an era-restricted mode,
searching only within the question's own era's passages (30 candidates, see `era_passage_idx`
above), versus a full-corpus mode searching all 180 passages, since you need to report both.

In [ ]:
# TODO: implement the retrieval evaluation described above. For each condition
# (era-restricted / full-corpus) and, later, for each difficulty tier, you'll want Recall@1,
# Recall@3, Recall@5 (see K_VALUES) and MRR (mean reciprocal rank of the correct passage) across
# the evaluated questions.


def recall_row(label, recall, mrr, n):
    """Provided, optional convenience: if you computed recall as a dict {k: recall@k for k in
    K_VALUES}, plus an mrr float and a question count n, this formats them into one row for a
    results table. Adapt it (or skip it) if your own return shape looks different."""
    row = {"n": n, **{f"R@{k}": round(recall[k], 3) for k in K_VALUES}, "MRR": round(mrr, 3)}
    return {"condition": label, **row}

In [15]:
# TODO: report Recall@1/3/5 and MRR for era-restricted vs. full-corpus retrieval, and the same
# broken down by difficulty tier for both conditions (4 small tables total, or however you'd
# like to organise it). recall_row above + pd.DataFrame can help build a table.

**Discussion (write answer here):** See assignment


## (b) Chunking Sensitivity (10%)

TODO: build the chunk collection, for every passage, split it into chunks, fetch
embeddings for all the chunks, and keep track of which passage each chunk came from (you'll need this to map a chunk hit back to a passage_id for evaluation).

In [ ]:
# TODO: split each passage's text into two roughly-equal chunks. Splitting at the
# midpoint sentence boundary is sufficient (don't cut a sentence in half), but you may use a
# different chunking scheme if you prefer, just document what you did.

In [ ]:
# TODO: re-run the same full-corpus retrieval evaluation as part (a), but searching over
# chunks instead of whole passages. A hit counts if any chunk belonging to the correct passage
# appears in the top-k ranked chunks, dedupe by passage first, don't let two chunks from the
# same passage each count as a separate rank position.

# TODO: compare whole-passage full-corpus retrieval (part (a)) against this chunked version in
# one table.

**Discussion (write answer here):** see assignment

## (c) Retrieved vs. Parametric Knowledge (10%)

Needs a local chat model (LM Studio) or the Colab + Hugging Face alternative, see the setup guide. Graded on your pipeline and your discussion of what the model actually did, not on whether every generated answer is correct, see the assignment brief's note on small-model output being expected to be imperfect


In [ ]:
RUN_TASK_C = True
GENERATION_SAMPLE_PER_ERA = 2  # >= 2 per era required by the assignment (12 pairs total)

# TODO: write a small wrapper that calls your local chat model
# (client.chat.completions.create, same pattern as get_embeddings above but for the
# /chat/completions endpoint) and returns the text of its reply. Handle the case where the model
# wraps its answer in <think>...</think> reasoning tags (strip them) if you're using a
# reasoning-style model.

# PROMPTS for you
# you will notice that rarely the language model will tell you that it doesn't know the answer :)
SYSTEM_NO_CONTEXT = (
    "You are a helpful assistant answering a trivia question. Answer in one or two sentences. "
    "If you do not know the answer, say so plainly rather than guessing."
)
SYSTEM_WITH_CONTEXT = (
    "You are a helpful assistant. Answer the question using ONLY the passages provided below. "
    "Cite the passage id(s) you used in square brackets, e.g. [ember-01]. If the passages do not "
    "contain the answer, say so plainly rather than guessing."
)

In [ ]:
# TODO: pick >= 2 matched real/fantasy pairs per era from baseline_questions
# (12 pairs total). For each pair, run THREE conditions and print/collect the results:
#   1. The real question, SYSTEM_NO_CONTEXT, no retrieved passages (parametric knowledge probe).
#   2. The fantasy question, SYSTEM_NO_CONTEXT, no retrieved passages (expect a refusal, since the
#      era is invented -- record whether it actually refuses or hallucinates instead).
#   3. The fantasy question again, SYSTEM_WITH_CONTEXT, with the passage(s) you retrieved for it
#      in part (a) inserted into the prompt (format: "[passage_id] passage text"), see the
#      assignment brief for exactly which retrieval condition and how many passages to use.
# Keep the transcripts (e.g. append dicts to a list, build a DataFrame) so you can include a
# handful of examples in your write-up.

**Discussion (write answer here):** See assignment

## Before you submit

- [ ] Part 1: accuracy matrices for part (a) naive and all part (b) conditions, forgetting numbers,
      buffer-size comparison, written discussion for (a)/(b)/(c)
- [ ] Part 2: Recall@k/MRR tables for part (a) (era-restricted vs. full-corpus, by tier) and
      part (b) (whole-passage vs. chunked), written discussion for (a)/(b)
- [ ] Part 2(c): transcripts (all three conditions, >= 3-4 pairs) + written discussion
- [ ] This notebook runs top to bottom without errors on a machine with LM Studio
      running locally -- restart the kernel and run all cells once before submitting to check.
- [ ] README explaining how to run your notebook, per the submission instructions
